# **Modulo 3: Advanced Machine Learning**
# **Examen de Certificacion: Intento 1**
# **En este examen, trabajar?s con un dataset de mayor complejidad**
# **Aplicando un an?lisis no supervisado utilizando K-means o PCA para identificar patrones ocultos y luego utilizando un modelo Perceptr?n Multicapa (MLP) para realizar predicciones.**


### **1.Limpieza de datos**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# 1. Cargar y explorar los datos
train_data = pd.read_csv('E:/Bootcamp/examendojo/examenlearning/train.csv')
test_data = pd.read_csv('E:/Bootcamp/examendojo/examenlearning/test.csv')

# Verificar si hay valores nulos
print("Valores nulos en los datos de entrenamiento:", train_data.isnull().sum().sum())
print("Valores nulos en los datos de prueba:", test_data.isnull().sum().sum())

# Preprocesamiento: Normalizaci?n
features_train = train_data.drop(columns=['Activity', 'subject'])
features_test = test_data.drop(columns=['Activity', 'subject'])

scaler = MinMaxScaler()
normalized_train = scaler.fit_transform(features_train)
normalized_test = scaler.transform(features_test)

# Convertir a DataFrame
normalized_train_df = pd.DataFrame(normalized_train, columns=features_train.columns)
normalized_test_df = pd.DataFrame(normalized_test, columns=features_test.columns)

# Adjuntar las columnas de 'Activity' y 'subject' de nuevo
normalized_train_df['Activity'] = train_data['Activity']
normalized_train_df['subject'] = train_data['subject']
normalized_test_df['Activity'] = test_data['Activity']
normalized_test_df['subject'] = test_data['subject']
def handle_missing_values(df):
    # Llenar con la media o eliminar filas/columnas, dependiendo del caso
    df.fillna(df.mean(), inplace=True)
    return df


### **2. An?lisis no supervisado con PCA o K-means**

In [ ]:
# Opci?n PCA
pca = PCA(n_components=2)
principal_components_train = pca.fit_transform(normalized_train_df.drop(columns=['Activity', 'subject']))

# Crear DataFrame para componentes principales
pca_train_df = pd.DataFrame(data=principal_components_train, columns=['PC1', 'PC2'])
pca_train_df['Activity'] = train_data['Activity']

# Visualizar PCA
plt.figure(figsize=(10,6))
for activity in pca_train_df['Activity'].unique():
    subset = pca_train_df[pca_train_df['Activity'] == activity]
    plt.scatter(subset['PC1'], subset['PC2'], label=activity, s=10)
plt.title('PCA - Componentes principales agrupados por actividad')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend()
plt.show()
# Opci?n K-means
kmeans = KMeans(n_clusters=6, random_state=42)
kmeans.fit(normalized_train_df.drop(columns=['Activity', 'subject']))
kmeans_labels = kmeans.labels_

# Visualizaci?n de K-means
plt.scatter(principal_components_train[:, 0], principal_components_train[:, 1], c=kmeans_labels, cmap='viridis', s=10)
plt.title('K-means Clustering en el espacio PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()

### **3. Modelado con MLP**


In [ ]:
# Codificar las etiquetas de la actividad
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_data['Activity'])
y_test = label_encoder.transform(test_data['Activity'])

# Dividir el conjunto de entrenamiento en entrenamiento y validaci?n
X_train, X_val, y_train, y_val = train_test_split(normalized_train, y_train, test_size=0.2, random_state=42)

# Configurar el modelo MLP
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, batch_size=32, learning_rate_init=0.001, random_state=42)

# Entrenar el modelo
mlp.fit(X_train, y_train)

### **4. Evaluaci?n del modelo**


In [ ]:
# Evaluar el modelo con el conjunto de validaci?n
y_pred = mlp.predict(X_val)

# M?tricas de evaluaci?n
print("Reporte de clasificaci?n:")
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))

# Matriz de confusi?n
conf_matrix = confusion_matrix(y_val, y_pred)
print("Matriz de confusi?n:\n", conf_matrix)

## Como podemos observar, el modelo es capaz de predecir correctamente la mayoria de las actividades fisicas

### **5.Visualizacion de las curvas**


In [ ]:
plt.plot(mlp.loss_curve_)
plt.title('Curva de p?rdida del MLP')
plt.xlabel('?pocas')
plt.ylabel('P?rdida')
plt.show()

Reflexi?n sobre los Resultados del An?lisis No Supervisado
Insights del An?lisis de Clusters y PCA:

Dicho analisis proporciono insights utiles acerca de las actividades, desde caminar, caminar por las escaleras y correr, en dicho contexto, caracter?sticas como la aceleraci?n, la frecuencia card?aca y otros datos sensoriales son loas mas importantes. El an?lisis de PCA nos ayuda a identificar cu?les de estas caracter?sticas son m?s influyentes en la variabilidad de los datos.

Discute el rendimiento del MLP y c?mo podr?a mejorarse:
El MLP se entrena con los datos normalizados y se eval?a utilizando un conjunto de validaci?n. El uso de m?tricas como el reporte de clasificaci?n y la matriz de confusi?n permite evaluar el rendimiento del modelo, el cual es adecuado para lograr predecir las distintas actividades fisicas
